# Phase 1: Data Acquisition & Pre-processing Pipeline

**AstroNova 2026: Deep Learning the Cosmic Web**  
**Objective:** Build a reliable, reproducible local/Colab data acquisition and pre-processing pipeline that transforms raw physical cosmological convergence maps into standard machine learning tensors following the methodologies of Liu et al. (2018) and Jeffrey et al. (2020).

### The 4 Steps of Phase 1:
1. **Step 1.1: Bulk Map Ingestion:** Ingest MassiveNuS Model 00 full-resolution $512 \times 512$ FITS convergence maps.
2. **Step 1.2: Shape Noise Injection (Creating X):** Transform ground truth convergence $\kappa_{\text{true}}$ (Y) to complex shear components $(\gamma_1, \gamma_2)$ via flat-sky Kaiser–Squires Fourier kernels, and inject independent pixel-level Gaussian shape noise $\mathcal{N}(0, \sigma_{\text{pixel}}^2)$ matching weak lensing survey specifications.
3. **Step 1.3: Normalization & Tensor Preparation:** Scale datasets using a global scale factor (10.0) to step up tiny decimal values (typically $-0.05$ to $+0.15$) to standard ML tensor range, formatting channels into $(C, 512, 512)$ and $(1, 512, 512)$.
4. **Step 1.4: Dataset Splitting (Zero Data Leakage):** Strictly partition maps into Training (70%), Validation (15%), and Testing (15%) subsets grouped by simulation line-of-sight realization (`_XXXXr.fits`), ensuring maps from the same underlying realization are never mixed between splits.

**References:**
- Liu, J., et al. (2018). *MassiveNuS: Cosmological massive neutrino simulations*. JCAP, 03, 049.
- Kaiser, N., & Squires, G. (1993). *Mapping the dark matter with weak gravitational lensing*. ApJ, 404, 441.
- Jeffrey, N., et al. (2020). *DeepMass: Deep learning dark matter map reconstructions from DES SV weak lensing data*. MNRAS, 492, 5023–5029.
- Gatti, M., et al. (2021). *Dark Energy Survey Year 3 results: Weak lensing shape catalogue*. MNRAS, 504, 4312–4336.

## 1. Imports & Configuration (Colab & Local Drive Resolution)

The pipeline settings are defined via a unified YAML configuration. Set `PHASE1_CONFIG` to a specific YAML path or edit `CONFIG_PATH_OVERRIDE` below. `DATASET_ROOT` is the primary path teammates customize; relative `INPUT_DIR` and `OUTPUT_DIR` are resolved from it.

In [ ]:
import csv
import glob
import os
import re
import tarfile
import time
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml
from astropy.io import fits
from scipy import fft as sfft

import torch
from torch.utils.data import DataLoader, Dataset

# Check hardware acceleration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"Using compute device: {device}")


In [ ]:
# Override with PHASE1_CONFIG=/path/to/config.yaml or set path below
CONFIG_PATH_OVERRIDE = None

DEFAULT_CONFIG_YAML = """
# Google Drive mount path for Google Colab
DATASET_ROOT: /content/drive/MyDrive/Astronova/Research/DataSet
INPUT_TAR: convergence_gal_mnv0.00000_om0.30000_As2.1000.tar
INPUT_DIR: convergence_gal_mnv0.00000_om0.30000_As2.1000/Maps05
OUTPUT_DIR: generated_data/phase1_sample_1

# Survey & Shape Noise Parameters (DES SV / MassiveNuS baseline parameters)
N_GAL: 6.0                      # Galaxy density per arcmin^2 (6.0 for DES SV, 30 for LSST)
SIGMA_EPS: 0.28                 # Intrinsic shape noise (0.28 for DES SV, 0.30 for LSST)
A_PIXEL: 0.287                  # Angular pixel area in arcmin^2 (~0.287 for MassiveNuS 512x512)
SCALE_FACTOR: 10.0              # Fixed global scaling factor to step up tiny decimal values
TARGET_ZERO_MEAN: true          # Subtract full-map mean (DC mode cannot be recovered by lensing)

# Dataset Splitting (70% Train, 15% Val, 15% Test strictly by realization)
TRAIN_RATIO: 0.70
VAL_RATIO: 0.15
TEST_RATIO: 0.15
SEED: 42

# Optional Survey Mask (chip gaps + star holes)
INCLUDE_MASK: true
MASK_FRACTION: 0.15
STAR_RADIUS_PX: [3, 13]
GAP_HEIGHT_PX: [6, 12]

# DataLoader Streaming Parameters
BATCH_SIZE: 16
NUM_WORKERS: 2

QA_FIGURE: qa_phase1_preprocessing.png
"""

# Check Colab Drive mount
running_in_colab = Path("/content").is_dir()
drive_root = Path("/content/drive")
drive_dataset_root = drive_root / "MyDrive/Astronova/Research/DataSet"

if running_in_colab and not drive_dataset_root.is_dir():
    try:
        from google.colab import drive as colab_drive
        colab_drive.mount("/content/drive")
    except Exception as exc:
        print(f"Colab Drive mount notice: {exc}")

config_override = CONFIG_PATH_OVERRIDE or os.environ.get("PHASE1_CONFIG") or os.environ.get("FORWARD_MODEL_CONFIG")
if config_override:
    CONFIG_PATH = Path(os.path.expandvars(os.path.expanduser(str(config_override))))
elif drive_dataset_root.is_dir():
    CONFIG_PATH = drive_dataset_root / "config_phase1.yaml"
else:
    CONFIG_PATH = Path.cwd() / "configs" / "phase1" / "config.yaml"

default_cfg = yaml.safe_load(DEFAULT_CONFIG_YAML)

# Local filesystem candidate fallbacks if running outside Colab
if not drive_dataset_root.is_dir():
    candidate_roots = [
        Path.cwd().parents[1] / "DataSet",
        Path.cwd().parent / "DataSet",
        Path.cwd() / "DataSet",
        Path.cwd() / "data"
    ]
    for cr in candidate_roots:
        if cr.is_dir():
            default_cfg["DATASET_ROOT"] = str(cr.resolve())
            break
    default_cfg.update({
        "INPUT_TAR": "convergence_gal_mnv0.00000_om0.30000_As2.1000.tar",
        "INPUT_DIR": "Subset-20261023T061241Z-1-001/Subset/Maps05",
        "OUTPUT_DIR": "generated_data/phase1_sample_1"
    })

if not CONFIG_PATH.exists():
    CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
    CONFIG_PATH.write_text(yaml.safe_dump(default_cfg, sort_keys=False), encoding="utf-8")
    print(f"Initialized config: {CONFIG_PATH.resolve()}")

with CONFIG_PATH.open(encoding="utf-8") as fh:
    loaded_cfg = yaml.safe_load(fh) or {}
cfg = {**default_cfg, **loaded_cfg}

DATASET_ROOT = Path(os.path.expandvars(os.path.expanduser(str(cfg["DATASET_ROOT"]))))
if not DATASET_ROOT.is_absolute():
    DATASET_ROOT = (CONFIG_PATH.parent / DATASET_ROOT).resolve()
else:
    DATASET_ROOT = DATASET_ROOT.resolve()

def resolve_config_path(val: str) -> Path:
    p = Path(os.path.expandvars(os.path.expanduser(str(val))))
    return p.resolve() if p.is_absolute() else (DATASET_ROOT / p).resolve()

INPUT_TAR = str(cfg.get("INPUT_TAR", "convergence_gal_mnv0.00000_om0.30000_As2.1000.tar"))
INPUT_DIR = resolve_config_path(cfg["INPUT_DIR"])
OUTPUT_DIR = resolve_config_path(cfg["OUTPUT_DIR"])
N_GAL = float(cfg["N_GAL"])
SIGMA_EPS = float(cfg["SIGMA_EPS"])
A_PIXEL = float(cfg["A_PIXEL"])
SCALE_FACTOR = float(cfg["SCALE_FACTOR"])
TARGET_ZERO_MEAN = bool(cfg["TARGET_ZERO_MEAN"])
TRAIN_RATIO = float(cfg["TRAIN_RATIO"])
VAL_RATIO = float(cfg["VAL_RATIO"])
TEST_RATIO = float(cfg["TEST_RATIO"])
SEED = int(cfg["SEED"])
INCLUDE_MASK = bool(cfg["INCLUDE_MASK"])
MASK_FRACTION = float(cfg["MASK_FRACTION"])
BATCH_SIZE = int(cfg["BATCH_SIZE"])
NUM_WORKERS = int(cfg.get("NUM_WORKERS", 2))
QA_FIGURE = str(cfg["QA_FIGURE"])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "splits").mkdir(parents=True, exist_ok=True)

# 1. Zero-Copy Discovery: Check if MassiveNuS tar archive exists under DATASET_ROOT
tar_candidates = [
    DATASET_ROOT / INPUT_TAR,
    DATASET_ROOT / "convergence_gal_mnv0.00000_om0.30000_As2.1000.tar",
    *list(DATASET_ROOT.glob("*.tar"))
]
FOUND_TAR_PATH = next((t for t in tar_candidates if t.is_file()), None)

FITS_FILES = []
USE_TAR_STREAMING = False

if FOUND_TAR_PATH:
    # Read member list in-memory WITHOUT extracting or copying any files to disk
    print(f"Reading directly from archive: {FOUND_TAR_PATH.resolve()}")
    with tarfile.open(FOUND_TAR_PATH, "r:*") as tar:
        for m in tar.getmembers():
            if m.name.endswith((".fits", ".fit")) and ("Maps05" in m.name or "z0.50" in m.name):
                FITS_FILES.append(m.name)
    FITS_FILES.sort()
    if FITS_FILES:
        USE_TAR_STREAMING = True
        print(f"Indexed {len(FITS_FILES)} Maps05 convergence grids in-memory (Zero disk copying).")

if not FITS_FILES:
    # Fallback to direct directory scan if uncompressed files exist on disk
    search_dirs = [INPUT_DIR, DATASET_ROOT]
    fits_candidates = []
    for sd in search_dirs:
        if sd.is_dir():
            fits_candidates.extend(list(sd.rglob("WLconv_z0.50_*.fits")))
            if not fits_candidates:
                fits_candidates.extend(list(sd.rglob("*.fits")))

    seen = set()
    for f in fits_candidates:
        p_str = str(f.resolve())
        if p_str not in seen:
            seen.add(p_str)
            FITS_FILES.append(p_str)
    FITS_FILES.sort()

if not FITS_FILES:
    raise FileNotFoundError(
        f"No FITS maps found in {FOUND_TAR_PATH or INPUT_DIR}.\n"
        f"Please verify {INPUT_TAR} is placed in {DATASET_ROOT}."
    )

print("--- Phase 1 Configuration Initialized ---")
print(f"Config File       : {CONFIG_PATH.resolve()}")
print(f"Dataset Root      : {DATASET_ROOT}")
print(f"Source Mode       : {('In-Memory Tar Stream from ' + FOUND_TAR_PATH.name) if USE_TAR_STREAMING else ('Disk Directory: ' + str(INPUT_DIR))}")
print(f"Discovered Maps   : Found {len(FITS_FILES)} full-resolution convergence maps")
print(f"Output Directory  : {OUTPUT_DIR}")
print(f"Survey Parameters : n_gal={N_GAL} gal/arcmin^2 | sigma_eps={SIGMA_EPS} | A_pixel={A_PIXEL} arcmin^2")
print(f"Global Scaling    : scale_factor={SCALE_FACTOR} | zero_mean={TARGET_ZERO_MEAN}")
print(f"Splitting Ratios  : Train {TRAIN_RATIO*100:.0f}% | Val {VAL_RATIO*100:.0f}% | Test {TEST_RATIO*100:.0f}%")


## Step 1.1: Bulk Map Ingestion

We inspect and load the raw MassiveNuS convergence maps ($\kappa_{\text{true}}$) from the Model 00 (flat $\Lambda\mathrm{CDM}$) directory. Each file contains a $512 \times 512$ physical grid representing the projected dark matter mass density along that line of sight.

In [ ]:
# Inspect the primary HDU of a sample map (Direct in-memory read)
sample_map_ref = FITS_FILES[0]

if USE_TAR_STREAMING:
    with tarfile.open(FOUND_TAR_PATH, "r:*") as tar:
        f_obj = tar.extractfile(sample_map_ref)
        with fits.open(f_obj) as hdul:
            sample_data = hdul[0].data.astype(np.float32)
            sample_header = hdul[0].header
else:
    with fits.open(sample_map_ref) as hdul:
        sample_data = hdul[0].data.astype(np.float32)
        sample_header = hdul[0].header

print(f"Sample File : {os.path.basename(sample_map_ref)}")
print(f"Array Shape : {sample_data.shape} (Height, Width)")
print(f"Value Range : min = {sample_data.min():.5f}, max = {sample_data.max():.5f}")
print(f"Mean & Std  : mean = {sample_data.mean():.5f}, std = {sample_data.std():.5f}")
print(f"Finite Check: {np.isfinite(sample_data).all()} (no NaNs or Infs)")


## Step 1.2: Shape Noise Injection (Creating Observable X)

MassiveNuS provides noiseless convergence maps ($\kappa_{\text{true}}$), which serve as ground truth targets ($Y$). To simulate a real telescope survey, we convert convergence to complex shear $(\gamma_1, \gamma_2)$ using flat-sky Kaiser–Squires relations:
$$\tilde{\gamma}_1 = \frac{l_x^2 - l_y^2}{l_x^2 + l_y^2} \tilde{\kappa}, \qquad \tilde{\gamma}_2 = \frac{2 l_x l_y}{l_x^2 + l_y^2} \tilde{\kappa}$$

We then inject independent Gaussian shape noise per pixel component matching the survey specification:
$$\sigma_{\text{pixel}} = \frac{\sigma_\epsilon}{\sqrt{2 \cdot n_{\text{gal}} \cdot A_{\text{pixel}}}}$$
For DES SV baseline: $n_{\text{gal}} = 6.0$, $\sigma_\epsilon = 0.28$, $A_{\text{pixel}} = 0.287 \implies \sigma_{\text{pixel}} \approx 0.15088$.

In [ ]:
@lru_cache(maxsize=8)
def _nyquist_keep(shape: tuple[int, int]) -> np.ndarray:
    ny, nx = shape
    keep = np.ones((ny, nx // 2 + 1), dtype=np.float64)
    if ny % 2 == 0:
        keep[ny // 2, :] = 0.0
    if nx % 2 == 0:
        keep[:, -1] = 0.0
    keep.setflags(write=False)
    return keep

@lru_cache(maxsize=8)
def _ks_kernels(shape: tuple[int, int]) -> tuple[np.ndarray, np.ndarray]:
    ny, nx = shape
    ly = sfft.fftfreq(ny)[:, None]
    lx = sfft.rfftfreq(nx)[None, :]
    l2 = lx**2 + ly**2
    l2_safe = np.where(l2 == 0.0, 1.0, l2)
    d1 = (lx**2 - ly**2) / l2_safe
    d2 = (2.0 * lx * ly) / l2_safe
    d1[0, 0] = 0.0
    d2[0, 0] = 0.0
    keep = _nyquist_keep(shape)
    d1 = d1 * keep
    d2 = d2 * keep
    d1.setflags(write=False)
    d2.setflags(write=False)
    return d1, d2

def kaiser_squires_kappa_to_shear(kappa: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Converts a 2D convergence map into complex shear components (gamma1, gamma2)."""
    k = np.nan_to_num(np.asarray(kappa, dtype=np.float64), nan=0.0)
    d1, d2 = _ks_kernels(k.shape)
    k_ft = sfft.rfft2(k)
    g1 = sfft.irfft2(d1 * k_ft, s=k.shape)
    g2 = sfft.irfft2(d2 * k_ft, s=k.shape)
    return g1.astype(np.float32), g2.astype(np.float32)

# Calculate physical pixel noise standard deviation dynamically
sigma_pixel = float(SIGMA_EPS / np.sqrt(2.0 * N_GAL * A_PIXEL))
print(f"-> Galaxy Number Density (n_gal): {N_GAL} gal/arcmin^2")
print(f"-> Intrinsic Shape Noise (sigma_eps): {SIGMA_EPS}")
print(f"-> Angular Pixel Area (A_pixel): {A_PIXEL} arcmin^2")
print(f"-> Component Noise Std Dev (sigma_pixel): {sigma_pixel:.5f}")
print(f"-> Global Scale Factor: {SCALE_FACTOR}")

## Step 1.3: Normalization & Tensor Preparation

Cosmological convergence values are very small (typically fluctuating between $-0.05$ and $+0.15$). We apply the global `scale_factor = 10.0` (as in Jeffrey et al. 2020) to step up the values into an $\mathcal{O}(1)$ range for numerical stability in deep learning.

Channels are formatted into standard machine learning tensors:
- Input feature tensor $X$: Shape `(2, 512, 512)` containing $[\gamma_{1,\text{scaled}}, \gamma_{2,\text{scaled}}]$ (or `(3, 512, 512)` with mask).
- Target tensor $Y$: Shape `(1, 512, 512)` containing $[\kappa_{\text{scaled}}]$.

In [ ]:
def generate_survey_mask(shape: tuple[int, int], mask_fraction: float, rng: np.random.Generator = None) -> np.ndarray:
    """Generates a realistic binary telescope survey mask with a chip gap and star holes."""
    rng = rng if rng is not None else np.random.default_rng()
    h, w = shape
    mask = np.ones(shape, dtype=np.float32)
    if mask_fraction <= 0:
        return mask
    target = mask_fraction * h * w
    gap_h = max(1, int(round(rng.uniform(0.15, 0.35) * mask_fraction * h)))
    y0 = int(rng.integers(0, h - gap_h + 1))
    mask[y0:y0 + gap_h, :] = 0.0
    n_masked = gap_h * w
    r_min, r_max = max(2.0, 0.01 * min(h, w)), max(3.0, 0.05 * min(h, w))
    while n_masked < target:
        r = rng.uniform(r_min, r_max)
        cy, cx = rng.uniform(0, h), rng.uniform(0, w)
        ys = slice(max(int(cy - r), 0), min(int(cy + r) + 2, h))
        xs = slice(max(int(cx - r), 0), min(int(cx + r) + 2, w))
        yy, xx = np.ogrid[ys, xs]
        inside = (yy - cy)**2 + (xx - cx)**2 <= r**2
        region = mask[ys, xs]
        n_masked += int(region[inside].sum())
        region[inside] = 0.0
    return mask

class Phase1ShearDataset(Dataset):
    """PyTorch Dataset producing weak lensing shear observations and ground-truth convergence.
    Follows the forward modeling procedure in Jeffrey et al. (2020).
    Supports direct in-memory streaming from tar archive without copying files to disk.
    """
    def __init__(self, file_list, tar_path=None, n_gal=6.0, sigma_eps=0.28, A_pixel=0.287, scale_factor=10.0, zero_mean=True, include_mask=True, mask_fraction=0.15, fixed_noise=False, seed=42):
        self.file_list = sorted(file_list)
        self.tar_path = tar_path
        self.scale_factor = scale_factor
        self.zero_mean = zero_mean
        self.include_mask = include_mask
        self.mask_fraction = mask_fraction
        self.fixed_noise = fixed_noise
        self.seed = seed
        self.sigma_pixel = float(sigma_eps / np.sqrt(2.0 * n_gal * A_pixel))

    def __len__(self):
        return len(self.file_list)

    def _load_fits_data(self, idx):
        map_ref = self.file_list[idx]
        if self.tar_path:
            with tarfile.open(self.tar_path, "r:*") as tar:
                f_obj = tar.extractfile(map_ref)
                with fits.open(f_obj) as hdul:
                    return hdul[0].data.astype(np.float32)
        else:
            with fits.open(map_ref) as hdul:
                return hdul[0].data.astype(np.float32)

    def __getitem__(self, idx):
        # 1. Load Ground Truth Convergence Map (Y) directly into memory
        y_data = self._load_fits_data(idx)

        # 2. Subtract full-map mean (DC mode cannot be recovered by weak lensing shear)
        if self.zero_mean:
            y_data = y_data - float(np.mean(y_data))

        # 3. Convert convergence to true noiseless shear components via flat-sky Kaiser-Squires
        g1_true, g2_true = kaiser_squires_kappa_to_shear(y_data)

        # 4. Inject independent telescope shape noise per component
        rng = np.random.default_rng(self.seed + idx) if self.fixed_noise else np.random.default_rng()
        noise_g1 = rng.normal(0.0, self.sigma_pixel, size=y_data.shape).astype(np.float32)
        noise_g2 = rng.normal(0.0, self.sigma_pixel, size=y_data.shape).astype(np.float32)

        g1_obs = g1_true + noise_g1
        g2_obs = g2_true + noise_g2

        # 5. Apply Global Scale Factor
        g1_scaled = g1_obs * self.scale_factor
        g2_scaled = g2_obs * self.scale_factor
        y_scaled = y_data * self.scale_factor

        # 6. Format channels: X is (2, 512, 512) or (3, 512, 512) with survey mask
        if self.include_mask:
            mask = generate_survey_mask(y_data.shape, self.mask_fraction, rng=rng)
            g1_scaled = g1_scaled * mask
            g2_scaled = g2_scaled * mask
            X = torch.stack([
                torch.from_numpy(g1_scaled),
                torch.from_numpy(g2_scaled),
                torch.from_numpy(mask)
            ], dim=0)
        else:
            X = torch.stack([
                torch.from_numpy(g1_scaled),
                torch.from_numpy(g2_scaled)
            ], dim=0)

        Y = torch.from_numpy(y_scaled).unsqueeze(0)
        return X, Y


## Step 1.4: Dataset Splitting (Zero Data Leakage)

We strictly separate our maps into **Training (70%)**, **Validation (15%)**, and **Testing (15%)** subsets.
Crucially, maps originating from the same underlying simulation line-of-sight realization (`_XXXXr.fits`) are grouped and never mixed across splits to prevent cosmological data leakage.

In [ ]:
def execute_realization_splitting(file_list: list[str], output_dir: Path, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, seed=42):
    """Strictly partitions simulation maps by underlying line-of-sight realization ID.
    Prevents data leakage across Training, Validation, and Test subsets.
    """
    realization_to_files = {}
    pattern = re.compile(r"_(\\d{4})r(?:\\.fits)?$")

    for f in file_list:
        base_name = os.path.basename(f)
        match = pattern.search(base_name)
        rid = match.group(1) if match else "misc"
        realization_to_files.setdefault(rid, []).append(f)

    realizations = sorted(realization_to_files.keys())
    rng = np.random.default_rng(seed)
    shuffled_rids = rng.permutation(realizations).tolist()

    n_total = len(shuffled_rids)
    n_train = max(1, int(round(train_ratio * n_total)))
    n_val = max(1, int(round(val_ratio * n_total)))
    if n_train + n_val >= n_total and n_total > 2:
        n_val = max(1, n_total - n_train - 1)

    train_rids = set(shuffled_rids[:n_train])
    val_rids = set(shuffled_rids[n_train:n_train + n_val])
    test_rids = set(shuffled_rids[n_train + n_val:])

    splits = {"train": [], "val": [], "test": []}
    for rid, files in realization_to_files.items():
        if rid in train_rids:
            splits["train"].extend(files)
        elif rid in val_rids:
            splits["val"].extend(files)
        else:
            splits["test"].extend(files)

    for k in splits:
        splits[k].sort()

    splits_dir = output_dir / "splits"
    splits_dir.mkdir(parents=True, exist_ok=True)
    for split_name, s_files in splits.items():
        csv_path = splits_dir / f"{split_name}_manifest.csv"
        with csv_path.open("w", newline="", encoding="utf-8") as fh:
            writer = csv.writer(fh)
            writer.writerow(["filepath", "filename"])
            for sf in s_files:
                writer.writerow([sf, os.path.basename(sf)])

    return splits

splits = execute_realization_splitting(
    FITS_FILES,
    OUTPUT_DIR,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    seed=SEED
)

train_files = splits["train"]
val_files = splits["val"]
test_files = splits["test"]

# Instantiate Dataset objects with zero-copy tar streaming support
train_dataset = Phase1ShearDataset(
    train_files,
    tar_path=FOUND_TAR_PATH,
    n_gal=N_GAL,
    sigma_eps=SIGMA_EPS,
    A_pixel=A_PIXEL,
    scale_factor=SCALE_FACTOR,
    zero_mean=TARGET_ZERO_MEAN,
    include_mask=INCLUDE_MASK,
    mask_fraction=MASK_FRACTION,
    seed=SEED
)

val_dataset = Phase1ShearDataset(
    val_files,
    tar_path=FOUND_TAR_PATH,
    n_gal=N_GAL,
    sigma_eps=SIGMA_EPS,
    A_pixel=A_PIXEL,
    scale_factor=SCALE_FACTOR,
    zero_mean=TARGET_ZERO_MEAN,
    include_mask=INCLUDE_MASK,
    mask_fraction=MASK_FRACTION,
    seed=SEED + 1000
)

# Multi-process workers = 0 for in-memory tar streaming, NUM_WORKERS for filesystem
workers = 0 if USE_TAR_STREAMING else NUM_WORKERS
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=workers)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=workers)

print(f"\nRealization-Safe Dataset Partitions Built:")
print(f"-> Training maps     : {len(train_dataset)} ({len(train_loader)} batches/epoch)")
print(f"-> Validation maps   : {len(val_dataset)} ({len(val_loader)} batches/epoch)")
print(f"-> Test maps         : {len(test_files)}")


## Visual Verification & Tensor QA

Pull a single sample block from the training stream and verify:
- Input Channel 1: Noisy $\gamma_1$
- Input Channel 2: Noisy $\gamma_2$
- (Optional Channel 3: Survey Mask)
- Target: True Convergence $\kappa_{\text{true}}$
Verify tensor shapes: $X \in \mathbb{R}^{B \times C \times 512 \times 512}$, $Y \in \mathbb{R}^{B \times 1 \times 512 \times 512}$.

In [ ]:
sample_loader = DataLoader(train_dataset, batch_size=1, shuffle=False)
sample_X, sample_Y = next(iter(sample_loader))

# Extract arrays and strip batch/channel axes for plotting
g1_plot = sample_X[0, 0].numpy() / SCALE_FACTOR
g2_plot = sample_X[0, 1].numpy() / SCALE_FACTOR
kappa_target = sample_Y[0, 0].numpy() / SCALE_FACTOR

num_plots = 4 if INCLUDE_MASK else 3
fig, axes = plt.subplots(1, num_plots, figsize=(6 * num_plots, 5))

im0 = axes[0].imshow(g1_plot, cmap="seismic", vmin=-0.3, vmax=0.3)
axes[0].set_title(r"Input Channel 1: Noisy $\gamma_1$")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(g2_plot, cmap="seismic", vmin=-0.3, vmax=0.3)
axes[1].set_title(r"Input Channel 2: Noisy $\gamma_2$")
fig.colorbar(im1, ax=axes[1])

if INCLUDE_MASK:
    mask_plot = sample_X[0, 2].numpy()
    im_m = axes[2].imshow(mask_plot, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title("Input Channel 3: Survey Mask")
    fig.colorbar(im_m, ax=axes[2])

im_target = axes[-1].imshow(kappa_target, cmap="inferno", vmin=-0.05, vmax=0.15)
axes[-1].set_title(r"Target: True Convergence $\kappa_{true}$")
fig.colorbar(im_target, ax=axes[-1])

plt.tight_layout()
qa_fig_path = OUTPUT_DIR / QA_FIGURE
plt.savefig(qa_fig_path, dpi=180, bbox_inches="tight")
plt.show()

print(f"Verified Tensor Shapes -> Input X: {sample_X.shape} | Target Y: {sample_Y.shape}")
print(f"Saved Visual QA Figure: {qa_fig_path}")

## Phase 1 Artifacts & Manifest Export

We record the resolved configuration used for this run and export the master catalogue manifest.

In [ ]:
# Save resolved configuration
cfg_used_path = OUTPUT_DIR / "config_used.yaml"
with open(cfg_used_path, "w", encoding="utf-8") as fh:
    yaml.safe_dump(cfg, fh, sort_keys=False)

# Save master manifest
master_manifest = OUTPUT_DIR / "manifest.csv"
with open(master_manifest, "w", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    writer.writerow(["source_fits", "split", "realization"])
    for sname, sfiles in splits.items():
        for sf in sfiles:
            m = re.search(r"_(\d+)r\.fits", os.path.basename(sf))
            rid = m.group(1) if m else "unknown"
            writer.writerow([os.path.basename(sf), sname, rid])

print(f"=== Phase 1 Pipeline Successfully Completed ===")
print(f"Output Directory: {OUTPUT_DIR}")
for item in sorted(OUTPUT_DIR.rglob("*")):
    if item.is_file():
        print(f"  - {item.relative_to(OUTPUT_DIR)} ({item.stat().st_size / 1024:.1f} KB)")
print("\nDatasets are fully prepared and partitioned for downstream Phase 2 model training.")